In [1]:
import os
import tensorflow as tf
import json
from transformers import GPT2Tokenizer , TFGPT2LMHeadModel
import requests

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.padding_side="left"
model = TFGPT2LMHeadModel.from_pretrained("gpt2")

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Import Llama3

In [9]:
def generate_response_llama3(prompt,max_length=150):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
    "model":"llama3",
    "prompt":prompt,
    "stream": False,
    "temperature": 0.1,
    "options": {
        "num_predict":max_length
    }
        }
    )
    return response.json()["response"]

In [10]:
def generate_response_gpt2(input):
    #tokenized_input = tokenizer(input,return_tensors="tf",truncation=True)
    #prompt  = build_prompt(input)
    input_ids = tokenizer.encode(input,
                                return_tensors='tf',
                                max_length=512,
                                truncation=True)
    output_ids = model.generate(
        input_ids,
        num_beams=4,
        repetition_penalty=1.5,
        pad_token_id = tokenizer.eos_token_id,
        eos_token_id = tokenizer.eos_token_id,
        do_sample=False,
        #max_length=input_ids.shape[1] + 80,
        early_stopping=True,
        max_new_tokens=50
    )

    output = tokenizer.decode(output_ids[0],skip_special_tokens=True)
    response = output[len(input):].strip()

    # stop if model starts another User/Bot turn
    for stop_word in ["User:", "Bot:"]:
        if stop_word in response:
            response = response.split(stop_word)[0].strip()
    return response

In [11]:
def Prompt_generator(goal):
    modified_goal = goal.lower().strip()

    if "exam" in modified_goal :
        return [
            "Identify the Exam Topic",
            "Divide the topics into study sessions",
            "Review important concepts",
            "Practice with exercises or old exams",
            "Do a final revision"
        ]
    elif "project" in modified_goal :
        return [
            "Define the project goal",
            "Break the project into modules",
            "Implement one module at a time",
            "Test the implementation",
            "Improve the final result"
        ]
    elif "schedule" in modified_goal or "week" in modified_goal:
        return [
            "List the important tasks for the week",
            "Set priorities for each task",
            "Distribute tasks across days",
            "Reserve time for breaks and review"
        ]
    else :
        return [
            "Understand the goal",
            "Break the goal into small tasks",
            "Set priorities",
            "Start with the important task"
        ]

In [29]:
class Memory:
    def __init__(self):
        self.history=[]
        self.goal=""
    def add(self,role,content):
        self.history.append(f"{role}: {content}")
        self.history = self.history[-4:]
    def delete(self):
        self.history = []
        self.goal = []
    def setGoal(self,goal):
        self.goal = goal
    def getHistoryText(self):
        return "\n".join(self.history)

In [13]:
class TaskAgent:
    def run(self, user_input):
        steps = Prompt_generator(user_input)
        steps_text = "\n".join(steps)
        full_prompt = f"Goal: {user_input}\nSteps:\n{steps_text}\nUser:Follow the Instructions and give me response\nBot:"
        response = generate_response_llama3(full_prompt)
        return response

In [14]:
class DescriptiveAgent:
    def run(self,object_user):
        prompt = (
            f"Q: How to {object_user}?\n",
            f"A: To succeed, you need to",
        )
        description_response = generate_response_llama3(prompt)
        return "To Succeed you need to " + description_response

In [15]:
class EstimationAgent:
    def run(self,user_input):
        full_prompt = (
            f"TASK: {user_input}\n"
            f"Your job is ONLY to estimate the time for the task above.\n"
            f"Do NOT talk about exams or anything else.\n"
            f"Reply ONLY in this exact format:\n"
            f"- Minimum time: ?\n"
            f"- Average time: ?\n"
            f"- Maximum time: ?\n"
            f"Estimate:"
        )
        estimation_response = generate_response_gpt2(full_prompt)
        return estimation_response

In [32]:
class CoordinatorAgent:
    def __init__(self):
        self.task_agent = TaskAgent()
        self.descriptive_agent = DescriptiveAgent()
        self.estimation_agent = EstimationAgent()
        self.history_agent = Memory()

    def route(self, user_input):
        u = user_input.lower().strip()
        if "memory" in u :
            return self.history_agent.getHistoryText()
        elif any(word in u for word in ["task", "add task", "tasks", "step", "steps", "plan"]):
            print("[Coordinator] → TaskAgent")
            return self.task_agent.run(user_input)
        elif any(word in u for word in ["description","describe","define", "definition", "explain", "object"]):
            print("[Coordinator] → DescriptiveAgent")
            return self.descriptive_agent.run(user_input)
        elif any(word in u for word in ["estimate","estimation","period"]):
            print("[Coordinator] → EstimationAgent")
            return self.estimation_agent.run(user_input)
        else:
            return "I didn't understand. Please try again."

In [33]:
coordination = CoordinatorAgent()
while True:
  user_input = input("user :")
  if user_input == "exit":
    break
  response = coordination.route(user_input)
  print(response)

user : memory


user : exit
